In [1]:

#Import the libraries
import io
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, OneHotEncoder, OrdinalEncoder
from scipy.stats import chi2_contingency

#Core state manager
class DataState:
    """Stores the active dataframe and schema metadata."""
    def __init__(self, df=None):
        self.df = df
        self.numeric_cols = []
        self.categorical_cols = []
        if df is not None:
            self.refresh_schema()

    def refresh_schema(self, max_cat=15):
        """Identifies columns by type, ignoring high-cardinality noise."""
        for col in self.df.columns:
            converted = pd.to_numeric(self.df[col], errors='coerce')
            if not converted.isna().all():
                self.df[col] = converted

        self.numeric_cols = self.df.select_dtypes(include=[np.number]).columns.tolist()
        nominal = self.df.select_dtypes(exclude=[np.number]).columns
        self.categorical_cols = [c for c in nominal if self.df[c].nunique() <= max_cat]

#Pipeline operations
def clean_data(state, strategy='median'):
    """Imputes and de-duplicates the state dataframe."""
    df = state.df.copy()

    #Missing the values
    for col in df.columns:
        if df[col].isna().sum() > 0:
            fill_val = df[col].median() if col in state.numeric_cols else df[col].mode()[0]
            df[col] = df[col].fillna(fill_val)

    #Duplicates
    df = df.drop_duplicates()
    state.df = df
    return state

def get_ml_matrix(state, num_scaler='robust', cat_encoder='onehot'):
    """Transforms raw data into a normalized feature matrix."""

    #Scale the numbers
    scaler = RobustScaler() if num_scaler == 'robust' else StandardScaler()
    nums = pd.DataFrame(scaler.fit_transform(state.df[state.numeric_cols]),
                        columns=state.numeric_cols, index=state.df.index)

    #Encode categaries
    encoder = OneHotEncoder(sparse_output=False, drop='first')
    cats_raw = state.df[state.categorical_cols].astype(str)
    cats = pd.DataFrame(encoder.fit_transform(cats_raw),
                        columns=encoder.get_feature_names_out(), index=state.df.index)

    return pd.concat([nums, cats], axis=1)

#Visualization engine
class VisualEngine:
    """Handles all plotting logic independently of the cleaning state."""

    @staticmethod
    def plot_univariate(df, cols):
        for col in cols:
            fig = make_subplots(rows=1, cols=3, subplot_titles=('Violin', 'Scatter', 'Hist'))
            fig.add_trace(go.Violin(x=df[col], box_visible=True, name=col), row=1, col=1)
            fig.add_trace(go.Scatter(y=df[col], mode='markers', opacity=0.5), row=1, col=2)
            fig.add_trace(go.Histogram(x=df[col]), row=1, col=3)
            fig.update_layout(title=f"Profiling: {col}", height=400, showlegend=False)
            fig.show()

    @staticmethod
    def plot_associations(state):
        cols = state.numeric_cols + state.categorical_cols
        matrix = pd.DataFrame(index=cols, columns=cols, dtype=float)

        for c1 in cols:
            for c2 in cols:
                if c1 == c2:
                    matrix.loc[c1, c2] = 1.0
                elif c1 in state.numeric_cols and c2 in state.numeric_cols:
                    matrix.loc[c1, c2] = state.df[c1].corr(state.df[c2])
                else:
                    #Generic cramer's v/association proxy
                    obs = pd.crosstab(state.df[c1], state.df[c2])
                    if obs.size > 0:
                        chi2 = chi2_contingency(obs)[0]
                        matrix.loc[c1, c2] = np.sqrt(chi2 / (len(state.df) * (min(obs.shape)-1)))
                    else:
                        matrix.loc[c1, c2] = 0

        px.imshow(matrix, text_auto=".2f", color_continuous_scale='RdBu_r',
                  title="Association Heatmap").show()

#Execution
#1Load
url = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"
raw_df = pd.read_csv(url)
state = DataState(raw_df)

#2Process
clean_data(state)
ml_ready = get_ml_matrix(state)

#3Display(Same outputs as original)
print(f"Final Shape: {state.df.shape}")
display(ml_ready.head(5))

view = VisualEngine()
view.plot_univariate(state.df, ['Age', 'Fare'])
view.plot_associations(state)

Final Shape: (891, 12)


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Ticket,Fare,Sex_male,Embarked_Q,Embarked_S
0,-1.000000,0.0,0.0,-0.461538,1.0,0.0,0.000000,-0.312011,1.0,0.0,1.0
1,-0.997753,1.0,-2.0,0.769231,1.0,0.0,0.000000,2.461242,0.0,0.0,0.0
2,-0.995506,1.0,0.0,-0.153846,0.0,0.0,0.000000,-0.282777,0.0,0.0,1.0
3,-0.993258,1.0,-2.0,0.538462,1.0,0.0,-0.396270,1.673732,0.0,0.0,1.0
4,-0.991011,0.0,0.0,0.538462,0.0,0.0,0.444557,-0.277363,1.0,0.0,1.0
